# Smart City Paris: GHG Emissions Predictive Modeling
Dataset: silver.ghg_emissions.csv

## 1. Data Processing
* Loaded PySpark table into a Pandas DataFrame.
* Interpolated data to fill missing years for a continuous timeline.

## 2. Exploratory Data Analysis
* Trend Analysis: Bar and scatter plots showed a downward linear trend.
* Distribution: Histogram and box plots confirmed no extreme outliers.
* Correlation: Heatmap proved a strong correlation between time and emissions.
* Insights: Transport and Energy are the main local polluters. Outside supply chain emissions exceed local emissions.

## 3. Machine Learning Pipeline
* Algorithm: LinearRegression from sklearn.
* Feature X: year column.
* Target y: ghg_major_sectors_total column.
* Split: 80 percent Training, 20 percent Testing.

## 4. Model Evaluation
* R-Squared
* RMSE: Measured the average prediction error in tons.

## 5. Future Forecasting
* Created a DataFrame for years 2024 to 2030.
* Forecasted total emissions using the trained model.
* Saved the model accuracy metric directly into the final dataset.

In [0]:
# importing libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


In [0]:
# load the dataframe and transform  to pandas 
df=spark.table('workspace.silver.silver_ghg_emissions').toPandas()

In [0]:
# displaying the datasets 
df.head(5)

In [0]:
#  dataset shape 
df.shape

In [0]:
# columns type 
df.dtypes

In [0]:
# Identify Missing Values 
df.isnull().sum()

In [0]:
# statistical summary 
df.describe()

In [0]:
df

In [0]:
# Chronological Gap Interpolation - filling missing years data 
df = df.set_index('year').reindex(range(2004, 2024)).interpolate().reset_index()
df


In [0]:
# Emissions Over the Years ( bar chart )
plt.bar(df['year'],df['ghg_major_sectors_total'],color='teal')
plt.title('emissions _per_year')
plt.xlabel('year')
plt.ylabel('emissions')
plt.show()


In [0]:
# trend line 
plt.bar(df['year'], df['ghg_major_sectors_total'], color='teal', label='Emissions')

# Calculate the trend line

coefficients = np.polyfit(df['year'], df['ghg_major_sectors_total'], 1)
trend_function = np.poly1d(coefficients)

# Plot the trend line
plt.plot(df['year'], trend_function(df['year']), color='red', linestyle='--', linewidth=2, label='Trend Line')

# plot 
plt.title('emissions_per_year')
plt.xlabel('year')
plt.ylabel('emissions')
plt.legend()
plt.show()

In [0]:
# Box Plot 
plt.boxplot(df['ghg_major_sectors_total'])
plt.title("Outlier Check: Emissions")
plt.ylabel("Emissions (Tons)")
plt.show()

In [0]:
# correlation heatmap 
corr = df.select_dtypes('number').corr()

plt.imshow(corr, cmap='coolwarm')
plt.colorbar(label='Correlation (-1 to 1)')

# Add column names to the axes so we know what we are looking at
plt.xticks(ticks=range(len(corr.columns)), labels=corr.columns, rotation=90)
plt.yticks(ticks=range(len(corr.columns)), labels=corr.columns)

plt.title("Correlation Heatmap")
plt.show()

In [0]:
# A scatter plot reveals the exact shape of the trend.
plt.scatter(df['year'], df['ghg_major_sectors_total'], color='purple', s=50)
plt.title("Scatter Plot: Is the Trend Linear?")
plt.xlabel("Year")
plt.ylabel("Emissions (Tons)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
# Multi-Line Chart (Feature Exploration)
plt.plot(df['year'], df['ghg_major_sectors_total'], label='Total', marker='o', color='black')
plt.plot(df['year'], df['ghg_major_sectors_transport'], label='Transport', marker='s', color='red')
plt.plot(df['year'], df['ghg_major_sectors_energy'], label='Energy', marker='^', color='blue')

plt.title("Emissions by Sector Over Time")
plt.xlabel("Year")
plt.ylabel("Emissions (Tons)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
#  Which major sector is the biggest polluter?

sectors = [
    df['ghg_major_sectors_transport'].sum(), 
    df['ghg_major_sectors_energy'].sum(), 
    df['ghg_major_sectors_consumption'].sum()
]
labels = ['Transport', 'Energy', 'Consumption']

plt.pie(sectors, labels=labels, autopct='%1.1f%%', colors=['coral', 'lightblue', 'lightgreen'])
plt.title("Emissions Breakdown by Major Sector")
plt.show()

In [0]:
#  Line Chart: Inside Paris vs. Outside Paris

plt.plot(df['year'], df['ghg_local_details_total'], label='Inside Paris', marker='o', color='green')
plt.plot(df['year'], df['ghg_outside_paris_details_total'], label='Outside Paris (Supply Chain)', marker='s', color='red')

plt.title("Where do the emissions actually happen?")
plt.xlabel("Year")
plt.ylabel("Emissions (Tons)")
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

In [0]:
# What drives the local emissions inside Paris?

local_sources = [
    df['ghg_local_details_residential'].sum(),
    df['ghg_local_details_commercial'].sum(),
    df['ghg_local_details_transport'].sum(),
    df['ghg_local_details_waste'].sum()
]
source_labels = ['Residential', 'Commercial', 'Transport', 'Waste']

plt.bar(source_labels, local_sources, color='purple', edgecolor='black')
plt.title("Local Paris Emissions by Source")
plt.ylabel("Total Emissions (Tons)")
plt.show()

# Train and Evaluate the Model

In [0]:
# Define Features (X) and Target (y)
X = df[['year']] 
y = df['ghg_major_sectors_total']

In [0]:
# Split the data into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Initialize and Train the Model
model = LinearRegression()
model.fit(X_train, y_train)

In [0]:
# Make Predictions on test data and Evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- Model Evaluation ---")
print(f"R-squared Score: {r2:.2f}")
print(f"Root Mean Squared Error : {rmse:.2f} tons\n")

# Create the Future Predictions DataFrame

In [0]:
# Create the future dataframe
future_years = [2024, 2025, 2026, 2027, 2028, 2029, 2030]
pred_df = pd.DataFrame({'year': future_years})

In [0]:
# Predict future emissions and round to 2 decimals
pred_df['predicted_ghg_total'] = model.predict(pred_df[['year']]).round(2)

In [0]:
# Store the model's R2 accuracy directly in the dataframe
pred_df['model_accuracy_r2'] = round(r2, 2)

In [0]:
# Display the final table 
display(pred_df)